In [1]:
import os
os.environ["RAY_DEDUP_LOGS"] = "0"
os.environ["RAY_memory_monitor_refresh_ms"] = "0"

In [2]:
import ray
import autotsc.utils as utils
from sklearn.base import clone
import asyncio
import logging
import ray
import time

from aeon.classification.feature_based import (
    Catch22Classifier,
    SummaryClassifier,
)
from aeon.classification.base import BaseClassifier
import numpy as np

2025-11-21 14:21:55.171639: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import numpy as np
from sklearn.base import clone
import ray

@ray.remote(num_cpus=1)
def run_model_on_fold(model_clone, X, y, train_idx, valid_idx):
    X_train, y_train = X[train_idx], y[train_idx]
    X_valid, y_valid = X[valid_idx], y[valid_idx]
    model_clone.fit(X_train, y_train)
    proba = model_clone.predict_proba(X_valid)
    val_probs = list(zip(valid_idx, proba))
    return model_clone, val_probs

class RayCrossValidationWrapper(BaseClassifier):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.trained_models_ = []

    def _fit(self, X, y):
        raise NotImplementedError()

    def _predict(self, X):
        raise NotImplementedError()

    def _fit_predict_proba(self, X, y, cv_splits):
        n_classes = len(np.unique(y))
        oof_proba = np.zeros((len(y), n_classes))

        fold_tasks = []
        for train_idx, val_idx in cv_splits:
            model_clone = clone(self.model)
            task = run_model_on_fold.remote(model_clone, X, y, train_idx, val_idx)
            fold_tasks.append(task)

        results = ray.get(fold_tasks)
        for model, proba in results:
            self.trained_models_.append(model)
            for idx, p in proba:
                oof_proba[idx] = p
        return oof_proba

    def fit_predict_proba(self, X, y, cv_splits):
        X, y, single_class = self._fit_setup(X, y)
        y_proba = self._fit_predict_proba(X, y, cv_splits)
        self.is_fitted = True
        return y_proba

@ray.remote(num_cpus=1)
def run_model_wrapper(wrapper, X, y, folds):
    return wrapper.fit_predict_proba(X, y, cv_splits=folds)

with utils.ray_init_or_reuse(num_cpus=24):

    X_train, y_train, X_test, y_test = utils.load_dataset('Car')
    folds = utils.get_folds(X_train, y_train, n_splits=10)

    m1 = RayCrossValidationWrapper(Catch22Classifier(n_jobs=1))
    m2 = RayCrossValidationWrapper(SummaryClassifier(n_jobs=1))
    m3 = RayCrossValidationWrapper(Catch22Classifier(n_jobs=1))
    m4 = RayCrossValidationWrapper(SummaryClassifier(n_jobs=1))

    t1 = run_model_wrapper.remote(m1, X_train, y_train, folds)
    t2 = run_model_wrapper.remote(m2, X_train, y_train, folds)
    t3 = run_model_wrapper.remote(m3, X_train, y_train, folds)
    t4 = run_model_wrapper.remote(m4, X_train, y_train, folds)

    res1, res2, res3, res4 = ray.get([t1, t2, t3, t4])

    print(res1.shape, res2.shape, res3.shape, res4.shape)



(60, 4) (60, 4) (60, 4) (60, 4)


In [4]:
fsdfs=DFsdfsfd

NameError: name 'DFsdfsfd' is not defined

(pid=gcs_server) [2025-11-21 14:22:27,381 E 692700 692700] (gcs_server) gcs_server.cc:302: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2025-11-21 14:22:28,785 E 692854 692854] (raylet) main.cc:975: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(run_model_wrapper pid=692958) [2025-11-21 14:22:30,839 E 692958 693314] core_worker_process.cc:825: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(run_model_wrapper pid=692963) [2025-11-21 14:22:30,837 E 692963 693278] core_worker_process.cc:825: Failed to establish connection to t

In [ ]:
import asyncio
import numpy as np
from sklearn.base import clone

@ray.remote(num_cpus=1)
def run_model_on_fold(model_clone, X, y, train_idx, valid_idx):
    X_train, y_train = X[train_idx], y[train_idx]
    X_valid, y_valid = X[valid_idx], y[valid_idx]
    model_clone.fit(X_train, y_train)
    proba = model_clone.predict_proba(X_valid)
    val_probs = zip(valid_idx, proba)
    return model_clone, val_probs

class RayCrossValidationWrapper(BaseClassifier):
    def __init__(self, model):
        self.trained_models_ = []
        self.ray_tasks_ = []
        self.model = model
        super().__init__()

    def _fit(self, X, y):
        pass

    def _predict(self, X):
        pass

    async def _fit_predict_proba(self, X, y, cv_splits) -> np.ndarray:
        oof_proba = np.zeros((len(y), len(np.unique(y))))
        for train_idx, val_idx in cv_splits:
            model_clone = clone(self.model)
            task = run_model_on_fold.remote(model_clone, X, y, train_idx, val_idx)
            self.ray_tasks_.append(task)
        
        await asyncio.sleep(0)
        
        results = ray.get(self.ray_tasks_)
        for model, proba in results:
            self.trained_models_.append(model)
            for val_idx, p in proba:
                oof_proba[val_idx] = p
        return oof_proba

    async def fit_predict_proba(self, X, y, **kwargs) -> np.ndarray:
        X, y, single_class = self._fit_setup(X, y)
        y_proba = await self._fit_predict_proba(X, y, **kwargs)
        self.is_fitted = True
        return y_proba

    def _fit_predict(self, X, y, cv_splits):
        pass


ray.init(ignore_reinit_error=True, include_dashboard=False, num_cpus=50, _metrics_export_port=None, logging_level=logging.WARNING)

X_train, y_train, X_test, y_test = utils.load_dataset('Car')
folds = utils.get_folds(X_train, y_train, n_splits=10)

m1 = RayCrossValidationWrapper(Catch22Classifier(n_jobs=1))
m2 = RayCrossValidationWrapper(SummaryClassifier(n_jobs=1))
m3 = RayCrossValidationWrapper(Catch22Classifier(n_jobs=1))
m4 = RayCrossValidationWrapper(SummaryClassifier(n_jobs=1))

async def main():
    with utils.ray_init_or_reuse(num_cpus=24):
        task1 = asyncio.create_task(m1.fit_predict_proba(X_train, y_train, cv_splits=folds))
        task2 = asyncio.create_task(m2.fit_predict_proba(X_train, y_train, cv_splits=folds))
        task3 = asyncio.create_task(m3.fit_predict_proba(X_train, y_train, cv_splits=folds))
        task4 = asyncio.create_task(m4.fit_predict_proba(X_train, y_train, cv_splits=folds))
        
        res1 = await task1
        res2 = await task2
        res3 = await task3
        res4 = await task4
        print(res1.shape, res2.shape, res3.shape, res4.shape)
        print(res1)

try:
    loop = asyncio.get_running_loop()
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(main())
except RuntimeError:
    asyncio.run(main())

In [ ]:


ray.init(ignore_reinit_error=True, include_dashboard=False, num_cpus=50, _metrics_export_port=None, logging_level=logging.WARNING)

@ray.remote(num_cpus=3)
def process_task(task_id: int, function_id: int) -> str:
    time.sleep(3)
    result = f"Function-{function_id} Task-{task_id} completed"
    print(f"  [Worker] {result}")
    return result


async def function_1():
    print("\n[Function 1] Scheduling jobs...")
    job_refs = [process_task.remote(i, 1) for i in range(5)]
    print("[Function 1] ✓ Jobs scheduled, releasing control...")
    await asyncio.sleep(0)
    print("[Function 1] Collecting results...")
    results = ray.get(job_refs)
    print("[Function 1] ✓ Done!")
    return {'function_id': 1, 'results': results}


async def function_2():
    print("\n[Function 2] Scheduling jobs...")
    job_refs = [process_task.remote(i, 2) for i in range(5)]
    print("[Function 2] ✓ Jobs scheduled, releasing control...")
    await asyncio.sleep(0)
    print("[Function 2] Collecting results...")
    results = ray.get(job_refs)
    print("[Function 2] ✓ Done!")
    return {'function_id': 2, 'results': results}


async def function_3():
    print("\n[Function 3] Scheduling jobs...")
    job_refs = [process_task.remote(i, 3) for i in range(5)]
    print("[Function 3] ✓ Jobs scheduled, releasing control...")
    await asyncio.sleep(0)
    print("[Function 3] Collecting results...")
    results = ray.get(job_refs)
    print("[Function 3] ✓ Done!")
    return {'function_id': 3, 'results': results}


async def _async_main():
    print("="*60)
    print("Starting...")
    print("="*60)
    all_results = await asyncio.gather(function_1(), function_2(), function_3())
    print("\n" + "="*60)
    print("All completed!")
    print("="*60)
    return all_results


def main():
    try:
        loop = asyncio.get_running_loop()
        import nest_asyncio
        nest_asyncio.apply()
        return asyncio.run(_async_main())
    except RuntimeError:
        return asyncio.run(_async_main())


final_results = main()
ray.shutdown()

In [ ]:
sdfsdf=Sdfsdfs

In [ ]:
import ray
import time
import random
from aeon.classification.feature_based import (
    Catch22Classifier,
    SummaryClassifier,
)

@ray.remote(num_cpus=1)
def fake_job(x):
    # Sleep for 1-2 seconds
    print("Starting job with input:", x)
    time.sleep(random.uniform(4, 5))
    print("Finished job with input:", x)
    return x * x

@ray.remote(num_cpus=1)
def train_fold(classifier, X, y, folds):
    #selected_fold = folds.filter(pl.col("fold") == fold_id).to_dicts()[0]
    #train_idx = selected_fold["train_idx"]
    #test_idx = selected_fold["test_idx"]
    train_idx, test_idx = folds

    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]

    start_time = perf_counter()
    classifier.fit(X_train, y_train)
    end_time = perf_counter()
    training_time = end_time - start_time

    y_pred = classifier.predict(X_test)
    y_pred_zip = zip(test_idx, y_pred.tolist())

    y_prob = classifier.predict_proba(X_test)
    y_prob_zip = zip(test_idx, y_prob.tolist())

    return classifier, y_pred_zip, y_prob_zip, training_time, classifier.classes_

X_train, y_train, X_test, y_test = utils.load_dataset("PhalangesOutlinesCorrect")

folds = utils.get_folds(X_train, y_train, n_splits=10)

with utils.ray_init_or_reuse(num_cpus=3):
    class CrossValidationWrapper:
        def __init__(self, model):
            self.trained_models_ = []
            self.model = model
            super().__init__()

        def start_fit(self, X, y, folds):
            self.job = train_fold.remote(clone(self.model), X, y, folds[0])

        def collect_results(self):
            return ray.get(self.job)

    m1 = CrossValidationWrapper(Catch22Classifier())
    m2 = CrossValidationWrapper(Catch22Classifier())
    m3 = CrossValidationWrapper(Catch22Classifier())
    m4 = CrossValidationWrapper(Catch22Classifier())
    m5 = CrossValidationWrapper(Catch22Classifier())
    m6 = CrossValidationWrapper(Catch22Classifier())
    m_list = [m1, m2, m3, m4, m5, m6]
    for m in m_list:
        m.start_fit(X_train, y_train, folds)
    results = [m.collect_results() for m in m_list]
    print(results)